## Tests — Clean Helpers

Unit tests for the helper functions defined in `data_processing/clean.ipynb`.

Note: this notebook **imports** the helper code by reading the clean notebook and executing the helper cell source, so there is a single source of truth.


## Install dependencies (run once)

If you're running in a fresh environment, install required packages from `requirements.txt`.

In [1]:
from pathlib import Path

_root = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "data").is_dir())
!python3 -m pip install -r "{_root / 'requirements.txt'}"

## Load helper functions from `data_processing/clean.ipynb`

In [2]:
from __future__ import annotations

import json
import re
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path
from typing import Any

import pandas as pd


def find_project_root(start: Path | None = None) -> Path:
    """Find project root by locating the `data/` directory."""
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate project root containing a 'data/' directory")


def load_clean_helpers() -> dict[str, Any]:
    clean_nb = find_project_root() / "data_processing" / "clean.ipynb"
    nb = json.loads(clean_nb.read_text(encoding="utf-8"))
    helper_cell = None
    for cell in nb["cells"]:
        if cell.get("cell_type") != "code":
            continue
        src = "".join(cell.get("source", []))
        if "def parse_currency_to_float" in src and "class QaReport" in src:
            helper_cell = src
            break
    if helper_cell is None:
        raise RuntimeError(f"Could not find helpers cell in {clean_nb}")

    ns: dict[str, Any] = {
        "pd": pd,
        "re": re,
        "Any": Any,
        "datetime": datetime,
        "dataclass": dataclass,
        "Path": Path,
    }
    exec(helper_cell, ns, ns)
    return ns


ns = load_clean_helpers()
parse_currency_to_float = ns["parse_currency_to_float"]
parse_transaction_datetime = ns["parse_transaction_datetime"]
normalize_zip = ns["normalize_zip"]
normalize_mcc = ns["normalize_mcc"]
normalize_state = ns["normalize_state"]
yesno_to_bool = ns["yesno_to_bool"]
derive_is_online = ns["derive_is_online"]
dedupe_by_id = ns["dedupe_by_id"]
clean_text = ns["clean_text"]


## Unit tests

In [3]:
assert clean_text("  a   b  ") == "a b"
assert clean_text("   ") is None

assert parse_currency_to_float("$2,238 ") == 2238.0
assert parse_currency_to_float(" ") is None
assert parse_currency_to_float("NOT_MONEY") is None

assert parse_transaction_datetime("2010-01-01 00:07:00") == datetime(2010, 1, 1, 0, 7, 0)
assert parse_transaction_datetime("bad") is None

assert normalize_zip("10464.0") == "10464"
assert normalize_zip("") is None

assert normalize_mcc("5812.0") == "5812"
assert normalize_mcc("abc") is None

assert normalize_state(" ny ") == "NY"

assert yesno_to_bool("YES") is True
assert yesno_to_bool("No") is False
assert yesno_to_bool("maybe") is None

assert derive_is_online("Online Transaction", "ONLINE") is True
assert derive_is_online("Swipe Transaction", "Bronx") is False

df_test = pd.DataFrame({"id": ["1", "1", "2"], "x": [1, 2, 3]})
seen = set(["0"])
df_deduped, dropped = dedupe_by_id(df_test, id_col="id", seen_ids=seen)
assert set(df_deduped["id"].tolist()) == {"1", "2"}
assert dropped == 1

print("Unit tests: PASS")


Unit tests: PASS
